# CS 79F Module 2 Lab 2: Metric Selection Under Cost Pressure

Daz Arnaud. Source: the course file m2_metric_scenarios.csv, downloaded from Canvas on November 5, 2026. Run this notebook in the same folder as the CSV.

Repository URL: Add after publishing.

Prediction 1, recorded before calculation: I expected S3 accuracy to be higher than recall. I did not record an approximate gap or my own reason before calculation, so I will not add one after the fact.


## Step 1. S3 by hand

S3 has 310 true positives, 44 false positives, 26 false negatives, and 9,600 true negatives. The total is 9,980.

Accuracy = (310 + 9,600) / 9,980 = 9,910 / 9,980 = 0.9930.

Precision = 310 / (310 + 44) = 310 / 354 = 0.8757.

Recall = 310 / (310 + 26) = 310 / 336 = 0.9226.

F1 = 2(310) / (2(310) + 44 + 26) = 620 / 690 = 0.8986.

The computed values match the course columns within rounding. Accuracy is about 7.04 percentage points higher than recall. The directional part of my prediction was correct.


In [1]:
from pathlib import Path
import csv

path = Path("m2_metric_scenarios.csv")
with path.open(newline="", encoding="utf-8-sig") as source:
    rows = list(csv.DictReader(source))
print("Course rows:", len(rows))
print("Scenario IDs:", ", ".join(row["scenario_id"] for row in rows))


Course rows: 4
Scenario IDs: S1, S2, S3, S4


## Step 2. Compute all four metrics

The code uses the counts in the course CSV. It compares each result with the supplied rounded metric.


In [2]:
results = []
for row in rows:
    tp = int(row["true_positive"])
    fp = int(row["false_positive"])
    fn = int(row["false_negative"])
    tn = int(row["true_negative"])
    accuracy = (tp + tn) / (tp + fp + fn + tn)
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * tp / (2 * tp + fp + fn)
    cost = fp * int(row["dollar_cost_per_false_positive"]) + fn * int(row["dollar_cost_per_false_negative"])
    values = (accuracy, precision, recall, f1)
    names = ("accuracy", "precision", "recall", "f1")
    match = all(abs(value - float(row[name])) < 0.00005 for value, name in zip(values, names))
    results.append({"id": row["scenario_id"], "accuracy": accuracy, "precision": precision,
                    "recall": recall, "f1": f1, "cost": cost, "match": match})

print("ID  Accuracy  Precision  Recall  F1      CSV match")
for result in results:
    print(f"{result['id']}  {result['accuracy']:.4f}    {result['precision']:.4f}     "
          f"{result['recall']:.4f}  {result['f1']:.4f}  {result['match']}")


ID  Accuracy  Precision  Recall  F1      CSV match
S1  0.9900    0.8333     0.9524  0.8889  True
S2  0.9918    0.9266     0.7947  0.8556  True
S3  0.9930    0.8757     0.9226  0.8986  True
S4  0.9536    0.8000     0.8455  0.8221  True


## Step 3. Compare accuracy with error cost

Prediction 2, recorded before cost comparison: Yes. Accuracy should track lower cost.

Total error cost = false positives times cost per false positive + false negatives times cost per false negative. Cost rank 1 means highest cost.


In [3]:
cost_order = sorted(results, key=lambda result: result["cost"], reverse=True)
cost_rank = {result["id"]: rank for rank, result in enumerate(cost_order, start=1)}
print("ID  Accuracy  Error cost  Cost rank (1 highest)")
for result in results:
    print(f"{result['id']}  {result['accuracy']:.4f}    ${result['cost']:,}  {cost_rank[result['id']]}")
print("Highest accuracy:", max(results, key=lambda result: result["accuracy"])["id"])
print("Lowest error cost:", min(results, key=lambda result: result["cost"])["id"])


ID  Accuracy  Error cost  Cost rank (1 highest)
S1  0.9900    $500,008  1
S2  0.9918    $7,520  4
S3  0.9930    $27,140  3
S4  0.9536    $343,000  2
Highest accuracy: S3
Lowest error cost: S2


My second prediction was wrong. S3 has the highest accuracy at 0.9930 and a $27,140 error cost. S2 has lower accuracy at 0.9918 but the lowest error cost at $7,520.

## Step 4. Metric choices, written before looking at any answer key

S1: I would optimize recall. A missed sepsis case costs $250,000 in the scenario, while a false alert costs $1.

S2: I would optimize precision. Blocking legitimate mail costs $40 per error, while missed spam costs $12.

S3: I would optimize recall. Shipping a defective part costs $900, while unnecessary inspection costs $85.

S4: I would optimize recall. Missing a case costs $3,200, while unnecessary review costs $300.

The downloaded CSV has an instructor_key column, but its four rows are blank. I found no answer key to compare.


## Step 5. S4 threshold sweep

Each step adds four false positives and removes one false negative. False positives cost $300 each. False negatives cost $3,200 each.


In [4]:
sweep = []
for step in range(6):
    fp = 130 + 4 * step
    fn = 95 - step
    cost = fp * 300 + fn * 3200
    sweep.append((step, fp, fn, cost))
print("Step  FP   FN  Error cost")
for step, fp, fn, cost in sweep:
    print(f"{step}     {fp}  {fn}  ${cost:,}")
best = min(sweep, key=lambda item: item[3])
print("Lowest modeled cost: step", best[0])
print("Saving against step 0: $", sweep[0][3] - best[3], sep="")


Step  FP   FN  Error cost
0     130  95  $343,000
1     134  94  $341,000
2     138  93  $339,000
3     142  92  $337,000
4     146  91  $335,000
5     150  90  $333,000
Lowest modeled cost: step 5
Saving against step 0: $10000


## Reflection

The course requires me to write these answers in my own words without AI assistance. I will complete them after reviewing the output.

1. Compare my Step 3 prediction with the table. Quote accuracy and cost for at least two scenarios.

2. Explain why S1 accuracy near 0.99 hides the hospital risk. Use the actual counts and state what I would report to an administrator.

3. State the best S4 setting and $10,000 saving. Explain when the extra false positives would make this setting a poor operational choice.

AI use disclosure: I used an AI assistant to structure the code and calculations. I checked the computed metrics against the course CSV and the cost arithmetic against the raw counts. The predictions I supplied are recorded above. I will write the reflection myself.
